# 🇸🇩 التدريب العصبي التلقائي للهجة السودانية (One-Click Automated Fine-Tuning)
### تطوير أوزان نموذج XTTS-v2 ليتكلم باللكنة ومخارج الحروف السودانية الأصيلة
---
**تم التحديث لإلغاء الواجهة اليدوية المعقدة — التدريب والحفظ إلى Google Drive يتم الآن آلياً بالكامل:**
1. `▶` **الخلية 1**: تثبيت حزمة التدريب المتوافقة وربط Google Drive.
2. `▶` **الخلية 2**: تجهيز وتقسيم الـ **4,120+ عينة صوتية** تلقائياً للتدريب والتقييم.
3. `▶` **الخلية 3**: تشغيل التدريب المباشر (Auto-Train) وحفظ النموذج تلقائياً في Google Drive.
4. `▶` **الخلية 4**: تجربة وسماع الصوت السوداني المدرب!

## 1️⃣ الخلية الأولى: تثبيت حزمة التدريب وربط Google Drive
*(تستغرق دقيقة واحدة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

!pip install -q coqui-tts pandas soundfile

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('\n✅ تم تثبيت المحرك وربط Google Drive بنجاح!')

## 2️⃣ الخلية الثانية: تجهيز وتقسيم البيانات للتدريب والتقييم (Auto-Split)
*(تقرأ الـ 4,120+ عينة وتنشئ ملفات `metadata_train.csv` و `metadata_eval.csv` في مجلد المشروع)*

In [ ]:
import pandas as pd
import os

DATASET_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset'
METADATA_FILE = os.path.join(DATASET_DIR, 'metadata.csv')
TRAIN_CSV = os.path.join(DATASET_DIR, 'metadata_train.csv')
EVAL_CSV = os.path.join(DATASET_DIR, 'metadata_eval.csv')

df = pd.read_csv(METADATA_FILE, sep='|', on_bad_lines='skip').dropna()
# تصفية أي نصوص قصيرة جداً أو علامات موسيقية
df = df[df['text'].str.len() >= 8]

# خيار حجم التدريب: استخدام 8,000 عينة نقية وموزعة يعطي تدريباً فائق الجودة وسريعاً (~25 دقيقة)
# يمكنك تعيين MAX_SAMPLES = None لتدريب كامل الداتاسيت (55 ألف عينة)
MAX_SAMPLES = 8000
if MAX_SAMPLES and len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
else:
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
eval_df = df.iloc[split_idx:]

train_df.to_csv(TRAIN_CSV, sep='|', index=False)
eval_df.to_csv(EVAL_CSV, sep='|', index=False)

print(f'📊 إجمالي العينات النقية المعتمدة للتدريب: {len(df):,} عينة سودانية!')
print(f'🏋️‍♂️ عينات التدريب (Train): {len(train_df):,}')
print(f'🧪 عينات الاختبار (Eval): {len(eval_df):,}')
display(train_df.head(5))


## 3️⃣ الخلية الثالثة: إطلاق التدريب التلقائي وحفظ النموذج في Drive مباشرة
*(ستبدأ العملية آلياً وتظهر لك خطوات التدريب وتناقص نسبة الخطأ، وتحفظ النموذج النهائي مباشرة داخل مجلد المشروع على Google Drive)*

In [ ]:
%cd /content
!rm -rf xtts-finetune-webui
!git clone -q https://github.com/daswer123/xtts-finetune-webui.git
import sys, os, shutil, glob
sys.path.append('/content/xtts-finetune-webui')

# 🛠️ ترقيع تلقائي لتوافق السكربت مع coqui-tts 0.28+ (تغيّر الـ API)
gpt_train_file = '/content/xtts-finetune-webui/utils/gpt_train.py'
src = open(gpt_train_file, encoding='utf-8').read()
# 1) XttsAudioConfig انتقل إلى TTS.tts.models.xtts
src = src.replace(
    'from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig',
    'from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig\n'
    'try:\n    from TTS.tts.layers.xtts.trainer.gpt_trainer import XttsAudioConfig\n'
    'except ImportError:\n    from TTS.tts.models.xtts import XttsAudioConfig')
# 2) load_tts_samples أصبحت تستقبل config يحتوي حقل datasets بدلاً من القائمة مباشرة
src = src.replace(
    'train_samples, eval_samples = load_tts_samples(\n        DATASETS_CONFIG_LIST,',
    'config.datasets = DATASETS_CONFIG_LIST\n    train_samples, eval_samples = load_tts_samples(\n        config,')
open(gpt_train_file, 'w', encoding='utf-8').write(src)
assert 'config.datasets = DATASETS_CONFIG_LIST' in src, 'لم ينجح ترقيع load_tts_samples'
print('✅ تم ترقيع gpt_train.py للتوافق مع النسخة الحالية من coqui-tts')

from utils.gpt_train import train_gpt

DATASET_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset'
TRAIN_CSV = os.path.join(DATASET_DIR, 'metadata_train.csv')
EVAL_CSV = os.path.join(DATASET_DIR, 'metadata_eval.csv')
OUT_DRIVE_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model'
os.makedirs(OUT_DRIVE_DIR, exist_ok=True)

print('🚀 جاري بدء التدريب العصبي على كرت الشاشة وحفظ المخرجات في Google Drive مباشرة...')
speaker_file, config_file, checkpoint, vocab_file, trainer_out, speaker_ref = train_gpt(
    custom_model="",
    version="v2.0.2",
    language="ar",
    num_epochs=4,
    batch_size=2,
    grad_acumm=8,
    train_csv=TRAIN_CSV,
    eval_csv=EVAL_CSV,
    output_path=OUT_DRIVE_DIR
)

best_models = glob.glob(os.path.join(trainer_out, '**', 'best_model*.pth'), recursive=True)
if best_models:
    shutil.copy(max(best_models, key=os.path.getctime), os.path.join(OUT_DRIVE_DIR, 'model.pth'))
shutil.copy(config_file, os.path.join(OUT_DRIVE_DIR, 'config.json'))
shutil.copy(vocab_file, os.path.join(OUT_DRIVE_DIR, 'vocab.json'))
print('\n🎉 اكتمل التدريب العصبي وتم حفظ النموذج السوداني المطور بنجاح في Google Drive!')


## 4️⃣ الخلية الرابعة: تجربة وسماع الصوت السوداني المدرب فوراً
*(يقرأ النموذج المدرب من مجلد `fine_tuned_model` في Drive ويولد الجملة بالنبرة السودانية الأصيلة)*

In [ ]:
import torch
from TTS.api import TTS
from IPython.display import Audio, display
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
FINE_TUNED_MODEL_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model'
REF_WAV = '/content/drive/MyDrive/Sudanese_Voice_Project/corpus/references/clean_podcast_8s.wav'

if os.path.exists(os.path.join(FINE_TUNED_MODEL_DIR, 'model.pth')):
    print('🎉 جاري تحميل النموذج السوداني المدرب حديثاً...')
    xtts_tuned = TTS(model_path=FINE_TUNED_MODEL_DIR, config_path=os.path.join(FINE_TUNED_MODEL_DIR, 'config.json')).to(device)
else:
    print('⚠️ لم يكتمل حفظ model.pth بعد، جاري استخدام النموذج الأساسي.')
    xtts_tuned = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

test_phrase = 'يا زول مرحب بيك في استوديو الصوت السوداني، دحين كدا النطق بقى سمح بالحيل وظابط تمام.'
out_audio = '/content/drive/MyDrive/Sudanese_Voice_Project/cloned_sudanese.wav'

xtts_tuned.tts_to_file(
    text=test_phrase,
    speaker_wav=REF_WAV,
    language='ar',
    file_path=out_audio,
    temperature=0.6,
    repetition_penalty=2.0
)

print('✅ تم التوليد والحفظ في Drive بنجاح!')
display(Audio(out_audio, autoplay=True))
